In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

"""
TRANSFORMACIÓN DE DATOS METEOROLÓGICOS A FORMATO LONG
----------------------------------------------------
Entrada:
    202i.xlsx

Salida:
    28 ficheros XLSX, uno por municipio.

Cada registro de salida contiene:
    timestamp + identificadores + 7 variables meteorológicas.

Las observaciones marcadas como N se convierten en NaN.
Se conservan las columnas de calidad para facilitar auditorías.
"""

In [8]:


# ==========================================================
# 1. CONFIGURACIÓN
# ==========================================================
ARCHIVO_ENTRADA0 = Path("2020.xlsx")
ARCHIVO_ENTRADA1 = Path("2021.xlsx")
ARCHIVO_ENTRADA2 = Path("2022.xlsx")
ARCHIVO_ENTRADA3 = Path("2023.xlsx")
ARCHIVO_ENTRADA4 = Path("2024.xlsx")
ARCHIVO_ENTRADA5 = Path("2025.xlsx")
ARCHIVO_ENTRADA6 = Path("2026.xlsx")
CARPETA_SALIDA = Path("datos_municipios")

# df = pd.read_csv("2026.csv", sep=";", decimal=',')
HOJAS_EXCEL = 0  # Primera hoja del Excel

# Códigos de las siete magnitudes meteorológicas
MAGNITUDES = {
    81: "viento_velocidad",
    82: "viento_direccion",
    83: "temperatura",
    86: "humedad",
    87: "presion",
    88: "radiacion",
    89: "precipitacion"
}

# Columnas identificativas
COLUMNAS_ID = [
    "provincia",
    "municipio",
    "estacion"
]

# Crear carpeta de salida
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

In [9]:

# ==========================================================
# 2. LECTURA DEL EXCEL
# ==========================================================

print("Leyendo fichero Excel...")

df0 = pd.read_excel(
    ARCHIVO_ENTRADA0,
    sheet_name=HOJAS_EXCEL
)
df1 = pd.read_excel(
    ARCHIVO_ENTRADA1,
    sheet_name=HOJAS_EXCEL
)
df2 = pd.read_excel(
    ARCHIVO_ENTRADA2,
    sheet_name=HOJAS_EXCEL
)
df3 = pd.read_excel(
    ARCHIVO_ENTRADA3,
    sheet_name=HOJAS_EXCEL
)
df4 = pd.read_excel(
    ARCHIVO_ENTRADA4,
    sheet_name=HOJAS_EXCEL
)

df5 = pd.read_excel(
    ARCHIVO_ENTRADA5,
    sheet_name=HOJAS_EXCEL
)
df6 = pd.read_excel(
    ARCHIVO_ENTRADA6,
    sheet_name=HOJAS_EXCEL
)
df = pd.concat([df0, df1, df2, df3, df4, df5, df6])

# Normalizar nombres de columnas
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.lower()
)

# Comprobar columnas necesarias
columnas_requeridas = (
    COLUMNAS_ID
    + ["ano", "mes", "dia", "magnitud"]
    + [f"h{i:02d}" for i in range(1, 25)]
    + [f"v{i:02d}" for i in range(1, 25)]
)

faltantes = [
    c for c in columnas_requeridas
    if c not in df.columns
]

if faltantes:
    raise ValueError(
        f"Faltan columnas en el Excel: {faltantes}"
    )

# Convertir campos numéricos
for col in ["ano", "mes", "dia", "magnitud"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Mantener únicamente las siete magnitudes
df = df[
    df["magnitud"].isin(MAGNITUDES.keys())
].copy()

print(f"Filas originales seleccionadas: {len(df):,}")
# print(df.head())
# print(df.describe())


Leyendo fichero Excel...
Filas originales seleccionadas: 455,362


In [10]:
print(df.head())
print(df.tail())
print(df.describe())

   provincia  municipio  estacion  magnitud  punto_muestreo   ano  mes  dia  \
0         28        102         1        81  28102001_81_89  2020    1    1   
1         28        102         1        82  28102001_82_89  2020    1    1   
2         28        102         1        83  28102001_83_89  2020    1    1   
3         28        102         1        86  28102001_86_89  2020    1    1   
4         28        102         1        87  28102001_87_89  2020    1    1   

     h01 v01  ...    h20 v20    h21 v21    h22 v22    h23 v23    h24 v24  
0    1.1   V  ...    0.2   V    0.3   V    0.9   V    0.7   V    0.4   V  
1   15.0   V  ...  291.0   V    3.0   V  359.0   V   18.0   V   27.0   V  
2    2.7   V  ...    7.3   V    6.3   V    5.3   V    4.3   V    4.2   V  
3   86.0   V  ...   68.0   V   74.0   V   81.0   V   86.0   V   85.0   V  
4  946.0   V  ...  944.0   V  945.0   V  945.0   V  945.0   V  944.0   V  

[5 rows x 56 columns]
       provincia  municipio  estacion  magnitud  pun

In [11]:
# ==========================================================
# 3. TRANSFORMACIÓN DE LAS 24 HORAS A REGISTROS
# ==========================================================

print("Transformando las 24 columnas horarias...")

registros = []

for hora in range(1, 25):

    columna_hora = f"h{hora:02d}"
    columna_validez = f"v{hora:02d}"

    temporal = df[
        COLUMNAS_ID
        + ["ano", "mes", "dia", "magnitud"]
        + [columna_hora, columna_validez]
    ].copy()

    temporal.rename(
        columns={
            columna_hora: "valor",
            columna_validez: "validacion"
        },
        inplace=True
    )

    # Convertir valor meteorológico a numérico
    temporal["valor"] = pd.to_numeric(
        temporal["valor"],
        errors="coerce"
    )

    # Normalizar indicadores de calidad
    temporal["validacion"] = (
        temporal["validacion"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    # Construcción de la fecha base
    fecha_base = pd.to_datetime(
        dict(
            year=temporal["ano"],
            month=temporal["mes"],
            day=temporal["dia"]
        ),
        errors="coerce"
    )

    # h01 = 01:00, h02 = 02:00 ... h24 = 24:00.
    # h24 se representa como las 00:00 del día siguiente.
    temporal["timestamp"] = (
        fecha_base
        + pd.to_timedelta(hora, unit="h")
    )

    # Aplicar validación:
    # V = válido
    # N = no válido
    # Otros indicadores = valor no validado
    temporal.loc[
        temporal["validacion"] != "V",
        "valor"
    ] = np.nan

    # Traducir código de magnitud a nombre de variable
    temporal["variable"] = temporal["magnitud"].map(
        MAGNITUDES
    )

    # Conservar indicador de calidad
    temporal["calidad"] = temporal["validacion"]

    registros.append(
        temporal[
            COLUMNAS_ID
            + [
                "timestamp",
                "variable",
                "valor",
                "calidad"
            ]
        ]
    )

# Unir los 24 bloques horarios
df_long = pd.concat(
    registros,
    ignore_index=True
)

print(
    f"Registros horarios generados: "
    f"{len(df_long):,}"
)
print(df_long.head())
print(df_long.describe())

Transformando las 24 columnas horarias...
Registros horarios generados: 10,928,688
   provincia  municipio  estacion           timestamp          variable  \
0         28        102         1 2020-01-01 01:00:00  viento_velocidad   
1         28        102         1 2020-01-01 01:00:00  viento_direccion   
2         28        102         1 2020-01-01 01:00:00       temperatura   
3         28        102         1 2020-01-01 01:00:00           humedad   
4         28        102         1 2020-01-01 01:00:00           presion   

   valor calidad  
0    1.1       V  
1   15.0       V  
2    2.7       V  
3   86.0       V  
4  946.0       V  
        provincia     municipio      estacion                      timestamp  \
count  10928688.0  1.092869e+07  1.092869e+07                       10928688   
mean         28.0  7.806012e+01  2.882671e+00  2023-07-19 08:18:01.584322816   
min          28.0  5.000000e+00  1.000000e+00            2020-01-01 01:00:00   
25%          28.0  1.600000e+01 

In [12]:
print(df_long.head(20))

    provincia  municipio  estacion           timestamp          variable  \
0          28        102         1 2020-01-01 01:00:00  viento_velocidad   
1          28        102         1 2020-01-01 01:00:00  viento_direccion   
2          28        102         1 2020-01-01 01:00:00       temperatura   
3          28        102         1 2020-01-01 01:00:00           humedad   
4          28        102         1 2020-01-01 01:00:00           presion   
5          28        102         1 2020-01-01 01:00:00         radiacion   
6          28        102         1 2020-01-01 01:00:00     precipitacion   
7          28        120         1 2020-01-01 01:00:00  viento_velocidad   
8          28        120         1 2020-01-01 01:00:00  viento_direccion   
9          28        120         1 2020-01-01 01:00:00       temperatura   
10         28        120         1 2020-01-01 01:00:00           humedad   
11         28        120         1 2020-01-01 01:00:00           presion   
12         2

In [13]:
# ==========================================================
# 4. PIVOTAR LAS SIETE MAGNITUDES
# ==========================================================

print("Integrando las siete variables meteorológicas...")

# Identificador de cada observación temporal
CLAVES = [
    "timestamp",
    "provincia",
    "municipio",
    "estacion"
]

# Comprobar posibles duplicados
duplicados = df_long.duplicated(
    subset=CLAVES + ["variable"],
    keep=False
)

if duplicados.any():
    raise ValueError(
        "Hay observaciones duplicadas para una misma "
        "estación, fecha y magnitud. Revise el Excel "
        "antes de continuar."
    )

# Pivotar los valores
df_valores = df_long.pivot(
    index=CLAVES,
    columns="variable",
    values="valor"
).reset_index()

df_valores.columns.name = None

# Pivotar indicadores de calidad
df_calidad = df_long.pivot(
    index=CLAVES,
    columns="variable",
    values="calidad"
).reset_index()

df_calidad.columns.name = None

# Renombrar columnas de calidad
df_calidad.rename(
    columns={
        variable: f"calidad_{variable}"
        for variable in MAGNITUDES.values()
    },
    inplace=True
)

# Combinar variables y calidad
df_final = df_valores.merge(
    df_calidad,
    on=CLAVES,
    how="left"
)

# Garantizar la presencia de las siete variables
for variable in MAGNITUDES.values():
    if variable not in df_final.columns:
        df_final[variable] = np.nan

# Ordenar cronológicamente
df_final.sort_values(
    by=["municipio", "estacion", "timestamp"],
    inplace=True
)

df_final.reset_index(drop=True, inplace=True)


Integrando las siete variables meteorológicas...


In [14]:
# ==========================================================
# 5. GENERAR UN FICHERO POR MUNICIPIO
# ==========================================================

print("Generando los ficheros municipales... Borrar y cargar: 049 106 115 127 134")

municipios = sorted(
    df_final["municipio"].dropna().unique()
)

resumen = []

for municipio in municipios:

    datos_municipio = df_final[
        df_final["municipio"] == municipio
    ].copy()

    # Código municipal con tres dígitos
    codigo = str(int(municipio)).zfill(3)

    nombre_archivo = (
        f"municipioTT_{codigo}.csv"
    )

    nombre_excel = (
        f"municipioTT_{codigo}.xlsx"
    )

    nombre_sheet = (
        f"municipioTT_{codigo}"
    )

    ruta_salida = (
        CARPETA_SALIDA / nombre_archivo
    )

    # Exportar CSV
    # datos_municipio.to_csv(
    #    ruta_salida,
    #    index=False,
    #    encoding="utf-8-sig",
    #    date_format="%Y-%m-%d %H:%M:%S"
    #)

    with pd.ExcelWriter(
        nombre_excel,
        engine="openpyxl"
    ) as writer:

        datos_municipio.to_excel(
            writer,
            sheet_name=nombre_sheet,
            index=False
        )

    resumen.append({
        "municipio": codigo,
        "filas": len(datos_municipio),
        "estaciones": datos_municipio[
            "estacion"
        ].nunique(),
        "fichero": nombre_excel
    })

    print(
        f"Municipio {codigo}: "
        f"{len(datos_municipio):,} registros"
    )


Generando los ficheros municipales...
Municipio 005: 59,856 registros
Municipio 006: 59,904 registros
Municipio 007: 59,904 registros
Municipio 009: 59,904 registros
Municipio 013: 59,880 registros
Municipio 014: 59,592 registros
Municipio 016: 59,664 registros
Municipio 045: 59,904 registros
Municipio 047: 59,904 registros
Municipio 049: 59,904 registros
Municipio 058: 59,904 registros
Municipio 065: 59,904 registros
Municipio 067: 59,760 registros
Municipio 074: 59,904 registros
Municipio 080: 59,904 registros
Municipio 092: 59,736 registros
Municipio 102: 59,880 registros
Municipio 106: 31,368 registros
Municipio 115: 31,680 registros
Municipio 120: 59,904 registros
Municipio 123: 59,808 registros
Municipio 127: 30,504 registros
Municipio 133: 59,904 registros
Municipio 134: 31,872 registros
Municipio 148: 59,280 registros
Municipio 161: 59,760 registros
Municipio 171: 59,880 registros
Municipio 180: 59,904 registros


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [15]:
# ==========================================================
# 6. INFORME FINAL
# ==========================================================

df_resumen = pd.DataFrame(resumen)

df_resumen.to_csv(
    CARPETA_SALIDA / "resumen_municipios.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n======================================")
print("PROCESO FINALIZADO")
print("======================================")

print(
    f"Municipios procesados: {len(municipios)}"
)

print(
    f"Total de registros: {len(df_final):,}"
)

print(
    f"Ficheros generados: {len(municipios)}"
)

print(
    f"Carpeta de salida: {CARPETA_SALIDA.resolve()}"
)

print("\nResumen:")
print(df_resumen.to_string(index=False))



PROCESO FINALIZADO
Municipios procesados: 28
Total de registros: 1,561,272
Ficheros generados: 28
Carpeta de salida: /content/datos_municipios

Resumen:
municipio  filas  estaciones              fichero
      005  59856           1 municipioTT_005.xlsx
      006  59904           1 municipioTT_006.xlsx
      007  59904           1 municipioTT_007.xlsx
      009  59904           1 municipioTT_009.xlsx
      013  59880           1 municipioTT_013.xlsx
      014  59592           1 municipioTT_014.xlsx
      016  59664           1 municipioTT_016.xlsx
      045  59904           1 municipioTT_045.xlsx
      047  59904           1 municipioTT_047.xlsx
      049  59904           1 municipioTT_049.xlsx
      058  59904           1 municipioTT_058.xlsx
      065  59904           1 municipioTT_065.xlsx
      067  59760           1 municipioTT_067.xlsx
      074  59904           1 municipioTT_074.xlsx
      080  59904           1 municipioTT_080.xlsx
      092  59736           1 municipioTT_092.x

In [16]:
COLUMNAS_GRUPO = [
    "provincia",
    "municipio",
    "estacion"
]

VARIABLES = [
    "humedad",
    "precipitacion",
    "presion",
    "radiacion",
    "temperatura",
    "viento_direccion",
    "viento_velocidad"
]

informes = []
resumen = []

df = pd.DataFrame()


In [17]:
def lectura_ficheros_global():
  # ============================================================
  # 8. LECTURA DEL ARCHIVO
  # ============================================================
  global df
  print("Leyendo fichero Excel...")
  df = pd.read_excel(
      ARCHIVO_ENTRADA,
      sheet_name=nombre_sheet
  )

  df.columns = df.columns.str.strip().str.lower()

  # Reiniciar índices para asegurar una asignación correcta
  df = df.reset_index(drop=True)

  # Comprobar columnas necesarias
  columnas_necesarias = (
      COLUMNAS_GRUPO
      + ["timestamp"]
      + VARIABLES
      + [f"calidad_{v}" for v in VARIABLES]
  )

  faltantes = [
      c for c in columnas_necesarias
      if c not in df.columns
  ]

  if faltantes:
      raise ValueError(
          f"Faltan columnas necesarias: {faltantes}"
      )

  # Convertir timestamp a formato fecha
  df["timestamp"] = pd.to_datetime(
      df["timestamp"],
      errors="raise"
  )

  # Normalizar indicadores de calidad
  for variable in VARIABLES:
      columna_calidad = f"calidad_{variable}"

      df[columna_calidad] = (
          df[columna_calidad]
          .astype("string")
          .str.strip()
          .str.upper()
      )

      valores_calidad = set(
          df[columna_calidad].dropna().unique()
      )

      if not valores_calidad.issubset({"V", "N"}):
          raise ValueError(
              f"Indicadores de calidad inesperados "
              f"en {columna_calidad}: {valores_calidad}"
          )

  # Orden cronológico
  df = df.sort_values(
      COLUMNAS_GRUPO + ["timestamp"]
  ).reset_index(drop=True)

  # Comprobar duplicados temporales por estación
  if df.duplicated(
      subset=COLUMNAS_GRUPO + ["timestamp"]
  ).any():
      raise ValueError(
          "Existen timestamps duplicados para una estación."
      )
  # print(df.describe())
  print(f"FIN lectura_ficheros originales: {len(df):,}\n")

In [18]:
# ============================================================
# 9. FUNCIÓN DE IMPUTACIÓN
# ============================================================

def imputar_mismo_dia_hora(df, variable):

    columna_calidad = f"calidad_{variable}"
    columna_imputado = f"imputado_{variable}"
    print()
    print(f"INICIO Imputando variable: {variable} - "
          f"columna_calidad: {columna_calidad} - "
          f"columna_imputado: {columna_imputado}"
    )
    # Inicializar indicador de imputación
    df[columna_imputado] = False
    # print(df.describe())
    # Un dato válido debe tener calidad V y valor numérico
    validos = (
        df[columna_calidad].eq("V")
        & df[variable].notna()
    )

    # Construir diccionario de datos históricos válidos.
    # La clave contiene estación y timestamp exacto.
    datos_validos = {}

    for fila in df.loc[
        validos,
        COLUMNAS_GRUPO + ["timestamp", variable]
    ].itertuples(index=False, name=None):

        grupo = tuple(fila[:len(COLUMNAS_GRUPO)])
        fecha = pd.Timestamp(
            fila[len(COLUMNAS_GRUPO)]
        )
        valor = fila[len(COLUMNAS_GRUPO) + 1]

        clave = grupo + (fecha,)

        datos_validos[clave] = valor

    # Identificar datos ausentes o no válidos
    necesitan_imputacion = (
        ~validos
    )

    indices = df.index[necesitan_imputacion]

    # Número máximo de días que se pueden consultar
    rango_dias = (
        df["timestamp"].max().date()
        - df["timestamp"].min().date()
    ).days + 1

    total_imputados = 0
    dias_retroceso = []

    # Buscar un valor válido en el mismo día y hora
    # de días anteriores
    for indice in indices:

        fecha_actual = pd.Timestamp(
            df.at[indice, "timestamp"]
        )

        grupo_actual = tuple(
            df.loc[indice, COLUMNAS_GRUPO].tolist()
        )

        valor_encontrado = False

        for dias in range(1, rango_dias + 1):

            fecha_busqueda = (
                fecha_actual - pd.Timedelta(days=dias)
            )

            clave_busqueda = (
                grupo_actual + (fecha_busqueda,)
            )

            if clave_busqueda in datos_validos:

                valor = datos_validos[clave_busqueda]

                # Imputar el valor histórico validado
                df.at[indice, variable] = valor

                # Marcar el registro como imputado
                df.at[indice, columna_imputado] = True

                total_imputados += 1
                dias_retroceso.append(dias)

                valor_encontrado = True
                break

        if not valor_encontrado:

            raise ValueError(
                f"No se encontró un dato histórico válido "
                f"para la variable '{variable}', "
                f"estación {grupo_actual}, "
                f"timestamp {fecha_actual}. "
                f"No se puede completar sin inventar un valor."
            )

    # Resumen de la imputación
    return {
        "Variable": variable,
        "Registros_imputados": total_imputados,
        "Registros_originales_validos": int(validos.sum()),
        "Retroceso_maximo_dias": (
            max(dias_retroceso) if dias_retroceso else 0
        ),
        "Retroceso_medio_dias": (
            round(float(np.mean(dias_retroceso)), 2)
            if dias_retroceso else 0
        )
    }

In [19]:
def ejecutar_imputacion(df):
  # ============================================================
  # 10. EJECUTAR LA IMPUTACIÓN DE LAS SIETE VARIABLES
  # ============================================================
  global informes
  print("INICIO Ejecutar imputación...")
  print()
  for variable in VARIABLES:

      print(
          f"Procesando variable: {variable}"
      )
      # print(df.describe())

      resultado = imputar_mismo_dia_hora(
          df,
          variable
      )

      informes.append(resultado)

      print(
          f"Registros imputados: {resultado['Registros_imputados']}"
      )

      print(
          f"Registros originales válidos: "
          f"{resultado['Registros_originales_validos']}"
      )
      print()


  print("FIN Ejecutar IMPUTACION")


In [20]:
def comprobacion_resultados():
  # ============================================================
  # 11. COMPROBACIÓN DE RESULTADOS
  # ============================================================

  global informe
  # Comprobar que no quedan valores ausentes
  # en las siete variables meteorológicas

  for variable in VARIABLES:

      if df[variable].isna().any():

          raise ValueError(
              f"Quedan valores ausentes en {variable}."
          )

  # Comprobar que todos los valores imputados
  # están identificados mediante su indicador

  for variable in VARIABLES:

      columna_imputado = f"imputado_{variable}"

      if df[columna_imputado].isna().any():

          raise ValueError(
              f"Hay indicadores de imputación vacíos "
              f"en {columna_imputado}."
          )

  # Comprobar que no quedan celdas vacías en el DataFrame
  if df.isna().any().any():

      columnas_con_nulos = (
          df.columns[df.isna().any()].tolist()
      )

      raise ValueError(
          f"Existen celdas vacías en estas columnas: "
          f"{columnas_con_nulos}"
      )

  # Crear informe de resultados
  informe = pd.DataFrame(informes)


In [21]:
def exportar_resultados():
  # ============================================================
  # 6. EXPORTAR RESULTADOS A EXCEL
  # ============================================================

  with pd.ExcelWriter(
      ARCHIVO_SALIDA,
      engine="openpyxl"
  ) as writer:

      df.to_excel(
          writer,
          sheet_name="Datos_completados",
          index=False
      )

      informe.to_excel(
          writer,
          sheet_name="Informe_imputacion",
          index=False
      )

  print("\n======================================")
  print("PROCESO DE IMPUTACIÓN FINALIZADO")
  print("======================================")

  print(f"Archivo generado: {ARCHIVO_SALIDA}")

  print("\nInforme de imputación:")
  print(informe.to_string(index=False))

  print("\nComprobación final:")
  print("No quedan valores ausentes.")


In [22]:
# ==========================================================
# 7. LEER UN FICHERO POR MUNICIPIO
# ==========================================================

print("Leyendo los ficheros municipales...")
# print(municipios)
print()

for municipio in municipios:

    # Código municipal con tres dígitos
    codigo = str(int(municipio)).zfill(3)

    ARCHIVO_ENTRADA = (
        f"municipioTT_{codigo}.xlsx"
    )

    nombre_sheet = (
        f"municipioTT_{codigo}"
    )

    ARCHIVO_SALIDA = (
        f"municipioTT_{codigo}_completado.xlsx"
    )
    print(
        f"Municipio: {codigo} "
        f"Archivo: {ARCHIVO_ENTRADA} "
        f"Hoja: {nombre_sheet} "
        f"Salida: {ARCHIVO_SALIDA}\n"
    )
    # Usamos la función modificada que actualiza la variable global df
    lectura_ficheros_global()
    ejecutar_imputacion(df)
    comprobacion_resultados()
    exportar_resultados()

    print(f"FIN iteracion: {municipio} \n")


Leyendo los ficheros municipales...

Municipio: 005 Archivo: municipioTT_005.xlsx Hoja: municipioTT_005 Salida: municipioTT_005_completado.xlsx

Leyendo fichero Excel...
FIN lectura_ficheros originales: 59,856

INICIO Ejecutar imputación...

Procesando variable: humedad

INICIO Imputando variable: humedad - columna_calidad: calidad_humedad - columna_imputado: imputado_humedad
Registros imputados: 609
Registros originales válidos: 59247

Procesando variable: precipitacion

INICIO Imputando variable: precipitacion - columna_calidad: calidad_precipitacion - columna_imputado: imputado_precipitacion
Registros imputados: 601
Registros originales válidos: 59255

Procesando variable: presion

INICIO Imputando variable: presion - columna_calidad: calidad_presion - columna_imputado: imputado_presion
Registros imputados: 605
Registros originales válidos: 59251

Procesando variable: radiacion

INICIO Imputando variable: radiacion - columna_calidad: calidad_radiacion - columna_imputado: imputado_ra

FINAL